<a href="https://colab.research.google.com/github/UberSw/curso-introduccion-programacion-cientifica/blob/main/ejercicios/taller_evaluativo_02_funciones_y_poo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Taller evaluativo 02 — Funciones y POO**
## **Uber Herley Chaverra Silva**
**11 de octubre de 2026**

En este ejercicio se implementa la función **calibrar_lectura**, la cual modela la recta de calibración *(valor * pendiente + intercepto)* para corregir las mediciones del sensor frente al equipo patrón. La función cuenta con parámetros por defecto (*pendiente = 1.0* e *intercepto = 0.0*) para que no altere el valor original en caso de no especificarlos. Finalmente, se realizan pruebas para evaluar la lectura sin calibrar, la lectura calibrada con los parámetros del sensor SBC-A (pendiente 1.2 e intercepto -0.5) y una prueba cambiando únicamente el intercepto, imprimiendo los resultados en pantalla con *print()*.

In [104]:
def calibrar_lectura(valor, pendiente = 1.0, intercepto = 0.0):
    """Aplica una recta de calibración a un valor de lectura."""
    return (valor * pendiente) + intercepto

sin_calibrar = calibrar_lectura(10.0)
calibrada = calibrar_lectura(10.0, 1.2, -0.5)
con_intercepto = calibrar_lectura(10.0, intercepto = 2.0)

print("---------- RESULTADOS ----------\n")
print(f"Sin calibrar:   {sin_calibrar}")
print(f"Calibrada:      {calibrada}")
print(f"Con intercepto: {con_intercepto}")

---------- RESULTADOS ----------

Sin calibrar:   10.0
Calibrada:      11.5
Con intercepto: 12.0


## **Ejercicio 2 - ¿Es utilizable la lectura?**
En este ejercicio se declara la lista **lecturas_sbc_a** con los valores del *sensor SBC-A* y se define la función booleana **es_lectura_valida.** La función evalúa que una lectura no sea igual al centinela de falla (-999.0) y que se encuentre dentro del rango físico aceptable (entre 0 y 500.0 inclusive). Finalmente, se verifica su funcionamiento imprimiendo las pruebas para los casos solicitados.

In [105]:
lecturas_sbc_a = [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]

def es_lectura_valida(valor):
    """Determina si una lectura no es centinela y está en el rango físico válido."""
    return (valor != -999.0) and (0.0 <= valor <= 500.0)

print("---------- RESULTADOS ----------\n")
print(f"es_lectura_valida(25.0):   {es_lectura_valida(25.0)}")
print(f"es_lectura_valida(-999.0): {es_lectura_valida(-999.0)}")
print(f"es_lectura_valida(600.0):  {es_lectura_valida(600.0)}")
print(f"es_lectura_valida(0.0):    {es_lectura_valida(0.0)}")

---------- RESULTADOS ----------

es_lectura_valida(25.0):   True
es_lectura_valida(-999.0): False
es_lectura_valida(600.0):  False
es_lectura_valida(0.0):    True


## **Ejercicio 3 - Promedio de las lecturas útiles**
En este ejercicio se implementa la función **promedio_valido**, la cual recibe una lista de lecturas y usa del Ejercico 2 la función **es_lectura_valida** para filtrar los datos descartables. Con un bucle *for,* acumula las lecturas válidas y cuenta cuántas cumplen el criterio para calcular el promedio. En caso de que no existan lecturas utilizables, retorna *None* y así evitar una división entre cero. Finalmente, se evalúa y muestra con *print()* el promedio de la lista **lecturas_sbc_a** y el de una lista compuesta únicamente por valores centinela.

In [106]:
def promedio_valido(lecturas):
    """Calcula el promedio de las lecturas validas o devuelve None si no hay ninguna."""

    suma = 0.0
    cuenta = 0

    for valor in lecturas:
        if es_lectura_valida(valor):
            suma += valor
            cuenta += 1

    if cuenta == 0:
        return None
    return (suma / cuenta)

promedio_sbc_a = promedio_valido(lecturas_sbc_a)

print("---------- RESULTADOS ----------\n")
print(f"promedio_sbc_a: {promedio_sbc_a}")
print(f"promedio_valido([-999.0, -999.0]): {promedio_valido([-999.0, -999.0])}")

---------- RESULTADOS ----------

promedio_sbc_a: 27.0
promedio_valido([-999.0, -999.0]): None


## **Ejercicio 4 - Calibrar la serie con map, filter y lambda**
En este ejercicio se procesan las lecturas del sensor SBC-A utilizando las funciones **map, filter** y funciones **lambda.** Primero, se filtran las lecturas crudas para conservar únicamente las válidas con **es_lectura_valida.** A continuación, se aplica la recta de calibración a cada dato útil con map y **calibrar_lectura.** Finalmente, mediante un nuevo filter con una función lambda, se seleccionan y contabilizan las lecturas calibradas que superan la categoría Moderada (> 35.4 ug/m3).

In [107]:
pendiente_sbc_a = 1.2
intercepto_sbc_a = -0.5

lecturas_utiles = filter(es_lectura_valida, lecturas_sbc_a)

lecturas_calibradas = list(
    map(lambda x: calibrar_lectura(x, pendiente_sbc_a, intercepto_sbc_a), lecturas_utiles)
)

excedencias = len(list(filter(lambda x: x > 35.4, lecturas_calibradas)))

print("------------------- RESULTADOS -------------------\n")
print(f"lecturas_calibradas: {lecturas_calibradas}")
print(f"excedencias:         {excedencias}")

------------------- RESULTADOS -------------------

lecturas_calibradas: [11.5, 29.5, 47.5, 9.1, 61.9]
excedencias:         2


## **Ejercicio 5 - La clase Lectura**
En este ejercicio se define la clase **Lectura,** para estructurar mediciones individuales junto con su *unidad* y el *código* del **sensor emisor.** La clase incluye métodos para convertir su concentración a microgramos **a_microgramos** y para comprobar si corresponde a un dato faltante o centinela **es_dato_faltante.**

In [109]:
class Lectura:
    """Representa una medicion de PM2.5 con su unidad y sensor de origen."""

    def __init__(self, valor, unidad = "ug/m3", codigo_sensor = "sin asignar"):
        """Inicializa la lectura con valor, unidad y codigo del sensor."""
        self.valor = valor
        self.unidad = unidad
        self.codigo_sensor = codigo_sensor

    def a_microgramos(self):
        """Devuelve el valor de la lectura convertido a microgramos por metro cubico."""
        if self.unidad == "mg/m3":
            return (self.valor * 1000)
        return self.valor

    def es_dato_faltante(self):
        """Indica si el valor corresponde al centinela de dato faltante."""
        return (self.valor == -999.0)

lectura_campo = Lectura(0.0252, "mg/m3", "SBC-A")
lectura_faltante = Lectura(-999.0, codigo_sensor="SBC-A")
lectura_normal = Lectura(30.0)

print("**************************************************")
print("*                   RESULTADOS                   *")
print("**************************************************\n")
print(f"lectura_campo.a_microgramos():       {lectura_campo.a_microgramos()}")
print(f"lectura_faltante.es_dato_faltante(): {lectura_faltante.es_dato_faltante()}")
print(f"lectura_campo.es_dato_faltante():    {lectura_campo.es_dato_faltante()}")
print(f"lectura_normal.unidad:               {lectura_normal.unidad}")
print(f"lectura_normal.codigo_sensor:        {lectura_normal.codigo_sensor}")

**************************************************
*                   RESULTADOS                   *
**************************************************

lectura_campo.a_microgramos():       25.2
lectura_faltante.es_dato_faltante(): True
lectura_campo.es_dato_faltante():    False
lectura_normal.unidad:               ug/m3
lectura_normal.codigo_sensor:        sin asignar


## **Ejercicio 6 - La clase Sensor: validación y encapsulamiento**
En este ejercicio se implementa la clase **Sensor,** la cual encapsula los parámetros de calibración de un equipo (pendiente e intercepto) como atributos internos precedidos de guion bajo **(_pendiente, _intercepto),** protegiéndolos mediante validación temprana en `__init__` El constructor lanza un **ValueError** si el código del sensor está vacío o si la pendiente es menor o igual a cero. La clase ofrece el método **calibra,** que reutiliza la función **calibrar_lectura** del Ejercicio 1, y define representaciones técnicas (`__repr__`) y legibles (`__str__`).

In [111]:
class Sensor:
    """Modela un sensor con sus parametros de calibracion validados y encapsulados."""

    def __init__(self, codigo, pendiente = 1.0, intercepto =0.0 ):
        """Inicializa y valida el codigo y los parametros lineales del sensor."""
        if codigo == "":
            raise ValueError("¡ERROR! El codigo del sensor no puede ser un texto vacio")
        if pendiente <= 0.0:
            raise ValueError("¡ERROR! La pendiente de calibracion debe ser estrictamente mayor a cero")

        self.codigo = codigo
        self._pendiente = pendiente
        self._intercepto = intercepto

    def calibrar(self, valor):
        """Devuelve el valor calibrado reutilizando la funcion calibrar_lectura."""
        return calibrar_lectura(valor, self._pendiente, self._intercepto)

    def __repr__(self):
        """Devuelve la representacion tecnica exacta del objeto sensor."""
        return f"Sensor(codigo='{self.codigo}', pendiente={self._pendiente}, intercepto={self._intercepto})"

    def __str__(self):
        """Devuelve una descripcion legible del sensor con sus parametros."""
        return f"Sensor {self.codigo} (pendiente: {self._pendiente}, intercepto: {self._intercepto})"

sensor_a = Sensor("SBC-A", 1.2, -0.5)
sensor_b = Sensor("SBC-B")

print("--------------- RESULTADOS ---------------\n")
print("sensor_a:", sensor_a)
print("Lista de sensores:", [sensor_a, sensor_b])
print("Calibrar 10.0 con sensor_a:", sensor_a.calibrar(10.0))
print("Calibrar 10.0 con sensor_b:", sensor_b.calibrar(10.0))

--------------- RESULTADOS ---------------

sensor_a: Sensor SBC-A (pendiente: 1.2, intercepto: -0.5)
Lista de sensores: [Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5), Sensor(codigo='SBC-B', pendiente=1.0, intercepto=0.0)]
Calibrar 10.0 con sensor_a: 11.5
Calibrar 10.0 con sensor_b: 10.0


## **Ejercicio 7 - La clase Estacion: objetos que contienen objetos**
En este ejercicio se define la clase **Estacion** para modelar una estación de monitoreo que almacena una colección de objetos **Lectura** (composición). La clase encapsula su lista de lecturas bajo el atributo interno *lecturas* y expone métodos para añadir instancias **agregar** y obtener la serie de concentraciones homogeneizadas a microgramos **valores**, invocando el método **a_microgramos** de cada lectura individual. Finalmente, se puebla la estación EST-04 a partir de los registros de **datos_campana**y se muestran su representación técnica y sus valores convertidos.


In [113]:
class Estacion:
    """Modela una estacion de monitoreo ambiental que contiene lecturas."""

    def __init__(self, codigo, sector):
        """Inicializa la estacion con su codigo, sector y lista interna de lecturas."""
        self.codigo = codigo
        self.sector = sector
        self._lecturas = []

    def agregar(self, lectura):
        """Agrega una instancia de Lectura a la coleccion interna."""
        self._lecturas.append(lectura)

    def valores(self):
        """Devuelve la lista de concentraciones en ug/m3 de todas las lecturas."""
        return [lectura.a_microgramos() for lectura in self._lecturas]

    def __repr__(self):
        """Devuelve la representacion tecnica exacta de la estacion."""
        return f"Estacion('{self.codigo}', '{self.sector}', lecturas={len(self._lecturas)})"

datos_campana = [
    (10.0, "ug/m3"),
    (-999.0, "ug/m3"),
    (0.025, "mg/m3"),
    (40.0, "ug/m3"),
    (-999.0, "ug/m3"),
    (8.0, "ug/m3"),
    (52.0, "ug/m3"),
    (600.0, "ug/m3"),
]

estacion_oriente = Estacion("EST-04", "Oriente")

for valor, unidad in datos_campana:
    estacion_oriente.agregar(Lectura(valor, unidad, codigo_sensor="SBC-A"))

print("--------------- RESULTADOS ---------------\n")
print(f"estacion_oriente:            {estacion_oriente}")
print(f"estacion_oriente.valores()): {estacion_oriente.valores()}")

--------------- RESULTADOS ---------------

estacion_oriente:            Estacion('EST-04', 'Oriente', lecturas=8)
estacion_oriente.valores()): [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


## **Ejercicio 8 - Integración: el informe de la estación**
En este último ejercicio se integran los conceptos de programación funcional y programación orientada a objetos. Se define la función **categoria_pm2** para clasificar concentraciones según la escala oficial mediante una estructura condicional **f / elif / else.** Posteriormente, se implementa la clase **InformeEstacion,** la cual almacena internamente una instancia de **Estacion** y una de **Sensor**. Sus métodos reutilizan las funciones auxiliares desarrolladas previamente **es_lectura_valida**, **promedio_valido**, **categoria_pm25** y el método **calibrar** del sensor) para calcular el promedio calibrado y determinar su categoría, formateando el resultado final a través del método especial `__str__`.

In [116]:
def categoria_pm25(concentracion):
    """Clasifica una concentracion de PM2.5 segun la escala oficial."""
    if concentracion <= 12.0:
        return "Buena"
    elif concentracion <= 35.4:
        return "Moderada"
    elif concentracion <= 55.4:
        return "Dañina para grupos sensibles"
    else:
        return "Dañina para la salud"


class InformeEstacion:
    """Genera un reporte integral de calidad del aire vinculando estacion y sensor."""

    def __init__(self, estacion, sensor):
        """Inicializa el informe guardando la estacion y el sensor como atributos internos."""
        self._estacion = estacion
        self._sensor = sensor

    def promedio_calibrado(self):
        """Calcula el promedio de las lecturas utiles de la estacion tras ser calibradas."""
        valores_crudos = self._estacion.valores()
        valores_utiles = filter(es_lectura_valida, valores_crudos)
        valores_calibrados = list(map(self._sensor.calibrar, valores_utiles))
        return promedio_valido(valores_calibrados)

    def categoria(self):
        """Devuelve la categoria oficial correspondiente al promedio calibrado."""
        return categoria_pm25(self.promedio_calibrado())

    def __str__(self):
        """Devuelve el reporte formateado en una linea con codigo, promedio y categoria."""
        prom = round(self.promedio_calibrado(), 1)
        return f"{self._estacion.codigo}: {prom} ug/m3 - {self.categoria()}"

    # Reto de extension (opcional)
    def excedencias(self, umbral=35.4):
        """Cuenta cuantas lecturas calibradas superan un umbral dado."""
        valores_utiles = filter(es_lectura_valida, self._estacion.valores())
        valores_calibrados = list(map(self._sensor.calibrar, valores_utiles))
        return len(list(filter(lambda x: x > umbral, valores_calibrados)))

informe = InformeEstacion(estacion_oriente, sensor_a)

print(f"---------- RESULTADOS ----------\n")
print(f"informe:                      {informe}")
print(f"informe.promedio_calibrado(): {informe.promedio_calibrado()}")
print(f"Excedencias:                  {informe.excedencias()}")


---------- RESULTADOS ----------

informe:                      EST-04: 31.9 ug/m3 - Moderada
informe.promedio_calibrado(): 31.9
Excedencias:                  2


In [118]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert abs(calibrar_lectura(10.0, 1.2, -0.5) - 11.5) < 1e-9, "revise el Ejercicio 1"
assert calibrar_lectura(10.0) == 10.0, "sin parámetros no debe alterar la lectura"
assert abs(con_intercepto - 12.0) < 1e-9, "revise con_intercepto del Ejercicio 1"
assert es_lectura_valida(25.0) and not es_lectura_valida(-999.0), "revise el Ejercicio 2"
assert not es_lectura_valida(600.0), "600.0 excede el rango del sensor"
assert abs(promedio_sbc_a - 27.0) < 1e-9, "revise el promedio del Ejercicio 3"
assert promedio_valido([-999.0, -999.0]) is None, "sin lecturas válidas debe devolver None"
assert len(lecturas_calibradas) == 5, "deberían quedar 5 lecturas calibradas"
assert abs(lecturas_calibradas[0] - 11.5) < 1e-9, "revise la calibración del Ejercicio 4"
assert excedencias == 2, "deberían ser 2 lecturas sobre 35.4"
assert abs(lectura_campo.a_microgramos() - 25.2) < 1e-9, "revise a_microgramos"
assert lectura_faltante.es_dato_faltante() and not lectura_campo.es_dato_faltante()
assert repr(sensor_a) == "Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)", "revise __repr__ de Sensor"
assert hasattr(sensor_a, "_pendiente") and not hasattr(sensor_a, "pendiente"), "la pendiente debe ser interna"
assert abs(sensor_b.calibrar(10.0) - 10.0) < 1e-9, "revise los valores por defecto de Sensor"
assert repr(estacion_oriente) == "Estacion('EST-04', 'Oriente', lecturas=8)", "revise __repr__ de Estacion"
assert estacion_oriente.valores() == lecturas_sbc_a, "revise valores() del Ejercicio 7"
assert abs(informe.promedio_calibrado() - 31.9) < 1e-9, "revise el Ejercicio 8"
assert informe.categoria() == "Moderada", "revise categoria()"